In [574]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [575]:
data = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv"
!wget $data -O course_lead_scoring.csv

--2025-10-13 22:28:02--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.01s   

2025-10-13 22:28:02 (7.36 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



In [576]:
df = pd.read_csv('course_lead_scoring.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [577]:
# Check for missing values
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [578]:
	
df.head().T

,0,1,2,3,4
lead_source,paid_ads,social_media,events,paid_ads,referral
industry,NaN,retail,healthcare,retail,education
number_of_courses_viewed,1,1,5,2,3
annual_income,79450.0,46992.0,78796.0,83843.0,85012.0
employment_status,unemployed,employed,unemployed,NaN,self_employed
location,south_america,south_america,australia,australia,europe
interaction_count,4,1,3,1,3
lead_score,0.94,0.8,0.69,0.87,0.62
converted,1,0,1,0,1


In [579]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
 
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
 
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
 
df.head().T

,0,1,2,3,4
lead_source,paid_ads,social_media,events,paid_ads,referral
industry,NaN,retail,healthcare,retail,education
number_of_courses_viewed,1,1,5,2,3
annual_income,79450.0,46992.0,78796.0,83843.0,85012.0
employment_status,unemployed,employed,unemployed,NaN,self_employed
location,south_america,south_america,australia,australia,europe
interaction_count,4,1,3,1,3
lead_score,0.94,0.8,0.69,0.87,0.62
converted,1,0,1,0,1


In [580]:
# Convert annual_income to numeric
df.annual_income = pd.to_numeric(df.annual_income, errors='coerce')
# Fill missing values with 0 
df.annual_income = df.annual_income.fillna(0)
df.annual_income.isnull().sum()

np.int64(0)

In [581]:
# Step 1: Replace empty strings (and those with only whitespaces) with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# Step 2: Replace NaN with string 'NA' in all text columns
df = df.fillna('NA')
# Replace NaN with 0 in all numeric columns
df = df.fillna(0)

In [582]:
df.head(5)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NA,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NA,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [583]:
categorical =['industry', 'location', 'lead_source','employment_status']
numerical = ['number_of_courses_viewed', 'annual_income', 'interaction_count', 'lead_score']

In [584]:
df['industry'].mode()
# Question 1: What is the most frequent industry in the dataset? retail

0    retail
Name: industry, dtype: object

In [585]:
# Split the data into training and test sets (80% - 20%)
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [586]:
# Question 2

In [587]:
len(df_full_train), len(df_test)

(1169, 293)

In [588]:
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [589]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

In [590]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [591]:
y_full_train = df_full_train.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [592]:
df_full_train[categorical].nunique()

industry             8
location             8
lead_source          6
employment_status    5
dtype: int64

In [593]:
df_full_train.converted

98      1
1188    0
1407    1
1083    0
404     0
       ..
715     0
905     1
1096    0
235     1
1061    1
Name: converted, Length: 1169, dtype: int64

In [594]:
df_full_train.converted.value_counts()

converted
1    715
0    454
Name: count, dtype: int64

In [595]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.611634
0    0.388366
Name: proportion, dtype: float64

In [596]:
global_converted_rate = df_full_train.converted.mean()
round(global_converted_rate, 2)

np.float64(0.61)

In [597]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [598]:
idx = np.arange(len(df))
# Use seed 42
seed = 42
np.random.seed(seed)
np.random.shuffle(idx)

In [599]:
# interaction_count and lead_score
df_full_train['interaction_count'].corr(df_full_train['lead_score'])

np.float64(0.011290499650258736)

In [600]:
# number_of_courses_viewed and lead_score
df_full_train['number_of_courses_viewed'].corr(df_full_train['lead_score'])

np.float64(-0.010710522152641658)

In [601]:
# number_of_courses_viewed and interaction_count
df_full_train['number_of_courses_viewed'].corr(df_full_train['interaction_count'])

np.float64(-0.026417136351258818)

In [602]:
# annual_income and interaction_count
df_full_train['annual_income'].corr(df_full_train['interaction_count'])

np.float64(0.06896871371403927)

In [603]:
# Question 2 answer
# annual_income and interaction_count


In [604]:
# Question 3
# Calculate the mutual information score between y and other categorical variables in the dataset. Use the training set only.
# Round the scores to 2 decimals using round(score, 2).


In [605]:

for i in range(len(categorical)):
    for j in range(i + 1, len(categorical)):
        score = mutual_info_score(df_full_train[categorical[i]], df_full_train[categorical[j]])
        print(categorical[i], categorical[j])
        print(round(score, 2))  

industry location
0.03
industry lead_source
0.02
industry employment_status
0.01
location lead_source
0.01
location employment_status
0.02
lead_source employment_status
0.01


In [606]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)
 
mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi

industry             0.008173
location             0.001212
lead_source          0.024562
employment_status    0.012690
dtype: float64

In [607]:
mi.sort_values(ascending=False)

lead_source          0.024562
employment_status    0.012690
industry             0.008173
location             0.001212
dtype: float64

In [608]:
# Question 3 answer
# lead_source 0.0245

In [609]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [610]:
# Question 4
# Now let's train a logistic regression.
# Remember that we have several categorical variables in the dataset. 
# Include them using one-hot encoding.
# Fit the model on the training dataset.
# To make sure the results are reproducible across different versions of Scikit-Learn, 
# fit the model with these parameters:
# model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
# What accuracy did you get?

In [611]:
# turns each column into a dictionary --> but the result is not what we want here
# df_train[['industry', 'location', 'lead_source','employment_status']].iloc[:100].to_dict()
 
dicts = df_train[['industry', 'location', 'lead_source','employment_status']].iloc[:100].to_dict(orient='records')
dicts
 

[{'industry': 'manufacturing',
  'location': 'europe',
  'lead_source': 'events',
  'employment_status': 'unemployed'},
 {'industry': 'NA',
  'location': 'south_america',
  'lead_source': 'referral',
  'employment_status': 'student'},
 {'industry': 'healthcare',
  'location': 'europe',
  'lead_source': 'organic_search',
  'employment_status': 'unemployed'},
 {'industry': 'other',
  'location': 'south_america',
  'lead_source': 'paid_ads',
  'employment_status': 'employed'},
 {'industry': 'education',
  'location': 'south_america',
  'lead_source': 'paid_ads',
  'employment_status': 'unemployed'},
 {'industry': 'manufacturing',
  'location': 'NA',
  'lead_source': 'referral',
  'employment_status': 'student'},
 {'industry': 'other',
  'location': 'middle_east',
  'lead_source': 'paid_ads',
  'employment_status': 'student'},
 {'industry': 'healthcare',
  'location': 'south_america',
  'lead_source': 'social_media',
  'employment_status': 'unemployed'},
 {'industry': 'NA',
  'location': '

In [612]:

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer


dicts_full_train = df_full_train[categorical+numerical].to_dict(orient='records') 

dv = DictVectorizer(sparse=False)
dv.fit(dicts_full_train)
 
# from this dictionaries we get the feature matrix
X_full_train = dv.fit_transform(dicts_full_train)

 
# then we train a model on this feature matrix
y_full_train = df_full_train.converted.values 

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_full_train, y_full_train)
model.predict(X_full_train)
 
# do the same things for test data
dicts_test = df_test[categorical].to_dict(orient='records')
X_test = dv.transform(dicts_test)
 
# do the predictions
y_pred = model.predict_proba(X_test)[:, 1]
converted_decision = (y_pred >= 0.5)
(y_val == converted_decision).mean()


np.float64(0.42662116040955633)

In [613]:
# Question 4 answer 0.426


In [614]:
# Question 5
# Let's find the least useful feature using the feature elimination technique.
# Train a model using the same features and parameters as in Q4 (without rounding).
# Now exclude each feature from this set and train a model without it. 
# Record the accuracy for each model.
# For each feature, calculate the difference between the original accuracy 
# and the accuracy without the feature.

In [620]:

small = categorical[:3]
print(small)
 
# df_train[small].iloc[:10]
# df_train[small].iloc[:10].to_dict(orient='records')
dicts_train_small = df_train[small].to_dict(orient='records')
dicts_val_small = df_val[small].to_dict(orient='records')
 
dv_small = DictVectorizer(sparse=False)
dv_small.fit(dicts_train_small)
dv_small.get_feature_names_out()

X_train_small = dv_small.transform(dicts_train_small)
y_train_small = df_full_train.converted.values 

model_small = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model_small.fit(X_train_small, y_train_small)
model_small.predict(X_train_small)[0,1]
 
# do the same things for test data
dicts_test = df_test[categorical].to_dict(orient='records')
X_test = dv.transform(dicts_test)
 
# do the predictions
y_pred = model_small.predict_proba(X_test)[:, 1]
 
# compute accuracy
converted_decision = (y_pred >= 0.5)
(converted_decision == y_test).mean()




['industry', 'location', 'lead_source']


ValueError: Found input variables with inconsistent numbers of samples: [876, 1169]